In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# --------------------------------------------------
# 1. Load dataset
# --------------------------------------------------
df = pd.read_csv("../data/merged_dataset_organized_47photic.csv")

# --------------------------------------------------
# 1B. Extract metadata from NaN-depth rows (one per cast)
# --------------------------------------------------
metadata_cols = ["CAST_COUNT", "Cruise_ID", "Cruz_Sta", "Cast_ID", "Sta_ID",
                 "Distance", "Date", "Time", "Lat_Dec", "Lon_Dec",
                 "Ac_Line", "Bottom_D", "Secchi", "IntChl", "IntC14",
                 "TimeZone", "Visibility"]

# Get metadata from rows where Secchi is not null (the secchi measurement rows)
metadata = (
    df[df["Secchi"].notna()][metadata_cols]
    .drop_duplicates("CAST_COUNT")
)

# --------------------------------------------------
# 2. Define baseline depth
# --------------------------------------------------
BASELINE_DEPTH = 2
SUMMING_DEPTH = 20  # Sum from 2m to 20m (or photic depth, whichever is shallower)

photic_depths = df[df["PHOTIC_ZONE"] == True][["CAST_COUNT", "DEPTH"]].rename(
    columns={"DEPTH": "PHOTIC_DEPTH"}
).drop_duplicates("CAST_COUNT")

print(f"Photic depths found for {len(photic_depths)} casts")

# Join photic depth onto full dataframe
df_with_photic = df.merge(photic_depths, on="CAST_COUNT", how="left")

print("\nCalculating summed variables...")

# Filter to baseline → min(20m, photic_depth) range per cast
df_range = df_with_photic[
    (df_with_photic["DEPTH"] >= BASELINE_DEPTH) &
    (df_with_photic["DEPTH"] <= df_with_photic["PHOTIC_DEPTH"].clip(upper=SUMMING_DEPTH))
]

print(f"Rows in summing range: {len(df_range)}")

# Sum variables over this range
summed = df_range.groupby("CAST_COUNT").agg(
    XMISS_SUMMED=("XMISS", "sum"),
    CHL_A_SUMMED=("CHL_A", "sum"),
    PHAEO_SUMMED=("PHAEO", "sum"),
    ESTCHL_SUMMED=("ESTCHL_STACORR", "sum"),
    BAT_SUMMED=("BAT", "sum")
).reset_index()

print(f"Summed variables calculated for {len(summed)} casts")

# ============================================================================
# KEEP ONLY PHOTIC_ZONE == TRUE ROWS
# ============================================================================

print("\nFiltering to photic zone rows only...")
df_photic = df[df["PHOTIC_ZONE"] == True].copy()

print(f"Photic zone rows: {len(df_photic)}")

# ============================================================================
# MERGE SUMMED VARIABLES ONTO PHOTIC ROWS
# ============================================================================

print("\nMerging summed variables...")
df_final = df_photic.merge(summed, on="CAST_COUNT", how="left")

# ============================================================================
# MERGE METADATA (clean version from Secchi rows)
# ============================================================================

print("Merging metadata...")

# Drop the potentially sparse metadata columns from photic rows
metadata_to_drop = [c for c in metadata_cols[1:] if c in df_final.columns]
df_final = df_final.drop(columns=metadata_to_drop)

# Merge clean metadata
df_final = df_final.merge(metadata, on="CAST_COUNT", how="left")

# --------------------------------------------------
# 7A. Two-point Beer-Lambert K_PAR (original formula)
# --------------------------------------------------

# Baseline PAR at 2m
baseline_par = df[df["DEPTH"] == BASELINE_DEPTH][
    ["CAST_COUNT", "PAR"]
].rename(columns={"PAR": "PAR_BASELINE"}).drop_duplicates("CAST_COUNT")

df_final = df_final.merge(baseline_par, on="CAST_COUNT", how="left")

df_final["K_PAR"] = -np.log(
    df_final["PAR"] / df_final["PAR_BASELINE"]
) / df_final["DEPTH"]

df_final = df_final.drop(columns=["PAR_BASELINE"])  # remove intermediate column


# --------------------------------------------------
# 7B. Regression slope-based K_PAR_SLOPE
# --------------------------------------------------

kpar_slope_results = []

for cast_id, cast_df in df.groupby("CAST_COUNT"):

    # Get photic depth
    photic_row = cast_df[cast_df["PHOTIC_ZONE"] == True]
    if photic_row.empty:
        continue

    photic_depth = photic_row["DEPTH"].values[0]

    # Subset depths between 2m and photic depth
    subset = cast_df[
        (cast_df["DEPTH"] >= BASELINE_DEPTH) &
        (cast_df["DEPTH"] <= photic_depth)
    ].copy()

    subset = subset[subset["PAR"] > 0]

    if len(subset) < 2:
        continue

    subset["LOG_PAR"] = np.log(subset["PAR"])

    X = subset[["DEPTH"]].values
    y = subset["LOG_PAR"].values

    model = LinearRegression()
    model.fit(X, y)

    slope = model.coef_[0]

    kpar_slope_results.append({
        "CAST_COUNT": cast_id,
        "K_PAR_SLOPE": -slope
    })

kpar_slope_df = pd.DataFrame(kpar_slope_results)

df_final = df_final.merge(kpar_slope_df, on="CAST_COUNT", how="left")

# --------------------------------------------------
# 8. Save final dataset
# --------------------------------------------------
df_final.to_parquet("../data/Parquet/New_estchlsummed_batsummed.parquet", index=False)

print(df_final.head())

/Users/annali/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/annali/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Photic depths found for 1971 casts

Calculating summed variables...
Rows in summing range: 35863
Summed variables calculated for 1971 casts

Filtering to photic zone rows only...
Photic zone rows: 1971

Merging summed variables...
Merging metadata...
   ORD_OCC    CAST_ID         DATE_TIME_UTC         DATE_TIME_PST  LAT_DEC  \
0      1.0  9308_001d  1993-08-11T12:06:56Z  1993-08-11T04:06:56Z    -99.0   
1      6.0  9308_006d  1993-08-12T10:57:24Z  1993-08-12T02:57:24Z    -99.0   
2     11.0  9308_011d  1993-08-13T14:53:14Z  1993-08-13T06:53:14Z    -99.0   
3     14.0  9308_014d  1993-08-14T10:19:40Z  1993-08-14T02:19:40Z    -99.0   
4     18.0  9308_018d  1993-08-15T10:12:16Z  1993-08-15T02:12:16Z    -99.0   

   LON_DEC       STA_ID  LINE    STA  DEPTH  ...     Lon_Dec  Ac_Line  \
0    -99.0  093.3 026.7  93.3   26.7   29.0  ... -117.305000     93.3   
1    -99.0  093.3 045.0  93.3   45.0   47.0  ... -118.563333     93.3   
2    -99.0  093.3 080.0  93.3   80.0   59.0  ... -120.933333 